# Notebook 03: Retrieval and Reranking

## Overview

This notebook demonstrates the multi-search retrieval pipeline:

```
Question  -->  Understand  -->  Encode Multiple Queries  -->  Chroma Search
                                                               |
                                                    Deduplicate + Rerank
                                                               |
                                                       Top-K Results
```

**What you will see:**
- How multiple queries are generated from one question
- How Chroma returns candidates for each query
- How deduplication keeps the best score per chunk
- How lexical and intent-based boosts improve ranking

**Source files:**
- `src/rag_app/retrieval/search.py` - Core search and reranking
- `src/rag_app/retrieval/query_understanding.py` - Intent classification

## 1. Setup

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.rag_app import config
from src.rag_app.retrieval.search import (
    get_embedding_model,
    get_collection,
    search,
    correct_section_title,
    is_patient_information_question,
    expand_question,
)
from src.rag_app.retrieval.query_understanding import understand_question, keyword_score

## 2. Load Model and Collection

In [ ]:
model = get_embedding_model()
collection = get_collection()
print(f"Collection: {config.COLLECTION_NAME}")
print(f"Total chunks: {collection.count()}")

## 3. Multi-Query Encoding

One question generates multiple queries. Each is encoded with the `query:` prefix (E5's format for queries vs passages).

In [ ]:
question = "ايه العلاج بعد الجراحة؟"
understanding = understand_question(question, model)

print(f"Question: '{question}'")
print(f"Intent: {understanding['intent']}")
print(f"\nGenerated {len(understanding['queries'])} queries:")
for i, q in enumerate(understanding['queries'], 1):
    print(f"  {i}. {q}")

# Encode all queries
query_texts = [f"query: {text}" for text in understanding['queries']]
query_vectors = model.encode(
    query_texts,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=False,
)
print(f"\nEncoded {len(query_vectors)} query vectors ({query_vectors.shape[1]} dimensions each)")

## 4. Chroma Search

We retrieve `top_k * 4` candidates to have enough room for reranking.
Results are filtered to `content_type == "recommendation"` only.

In [ ]:
candidate_count = min(config.TOP_K * 4, collection.count())
results = collection.query(
    query_embeddings=query_vectors.tolist(),
    n_results=candidate_count,
    where={"content_type": "recommendation"},
    include=["documents", "metadatas", "distances"],
)

print(f"Retrieved {candidate_count} candidates per query")
print(f"Number of query result sets: {len(results['ids'])}")

## 5. Deduplication

The same chunk may appear in results from multiple queries. We keep the highest semantic score for each chunk.

In [ ]:
candidates = {}
for result_index in range(len(results["ids"])):
    for chunk_id, document, metadata, distance in zip(
        results["ids"][result_index],
        results["documents"][result_index],
        results["metadatas"][result_index],
        results["distances"][result_index],
    ):
        semantic_score = 1 - float(distance)
        existing = candidates.get(chunk_id)
        if existing and existing["score"] >= semantic_score:
            continue
        candidates[chunk_id] = {
            "chunk_id": chunk_id,
            "score": semantic_score,
            "document_name": metadata["document_name"],
            "page_number": metadata["page_number"],
            "section_title": correct_section_title(metadata["section_title"], document),
            "content_type": metadata.get("content_type", "unknown"),
            "source_url": metadata["source_url"],
            "text": document,
            "intent": understanding["intent"],
        }

print(f"Before dedup: {candidate_count * len(results['ids'])} total results")
print(f"After dedup:  {len(candidates)} unique chunks")

## 6. Reranking: Three Signals

The final score combines:
1. **Semantic score** (0-1) - cosine similarity from Chroma
2. **Lexical boost** (0-0.08) - keyword overlap between query and chunk
3. **Intent boost** (0-0.12) - bonus for chunks matching the detected intent

In [ ]:
import re

rows = list(candidates.values())
for row in rows:
    lexical = max(keyword_score(row["text"], text) for text in understanding["queries"])
    boost = 0.0
    if understanding["intent"] == "symptoms_referral" and row["chunk_id"].startswith("ng12-"):
        boost = 0.12
    elif understanding["intent"] == "newly_diagnosed_information":
        if re.match(r"^\s*-?\s*1\.2\.1\b", row["text"]):
            boost = 0.12
        elif re.match(r"^\s*-?\s*1\.2\.\d+", row["text"]):
            boost = 0.06
    row["rerank_score"] = row["score"] + (0.08 * lexical) + boost
    row["lexical_boost"] = 0.08 * lexical
    row["intent_boost"] = boost

rows.sort(key=lambda row: row["rerank_score"], reverse=True)

print(f"Reranking breakdown for top candidates:")
print(f"{'Chunk ID':<25} {'Semantic':>8} {'Lexical':>8} {'Intent':>8} {'Final':>8}")
print("-" * 60)
for row in rows[:10]:
    print(
        f"{row['chunk_id']:<25} "
        f"{row['score']:>8.4f} "
        f"{row['lexical_boost']:>8.4f} "
        f"{row['intent_boost']:>8.4f} "
        f"{row['rerank_score']:>8.4f}"
    )

## 7. Final Results

In [ ]:
top_results = rows[:config.TOP_K]
for rank, row in enumerate(top_results, 1):
    row["rank"] = rank
    print(f"\n{'=' * 80}")
    print(f"Rank: {rank}")
    print(f"Semantic score: {row['score']:.4f}")
    print(f"Rerank score: {row['rerank_score']:.4f}")
    print(f"Document: {row['document_name']}")
    print(f"Section: {row['section_title']}")
    print(f"Page: {row['page_number']}")
    print(f"Chunk ID: {row['chunk_id']}")
    print(f"Source: {row['source_url']}")
    print("-" * 80)
    print(row["text"].strip()[:300])

## 8. Using the Full Search Function

The `search()` function wraps all of the above into a single call.

In [ ]:
# English question
results = search("What follow-up is recommended after surgery?")

print(f"Question: 'What follow-up is recommended after surgery?'")
print(f"Intent: {results[0]['intent']}")
print(f"\nTop {len(results)} results:")
for row in results:
    print(f"  #{row['rank']} (score: {row['rerank_score']:.4f}) Page {row['page_number']}: {row['section_title']}")
    print(f"     {row['text'][:100]}...")

## 9. Section Title Correction

A PDF page can end with the next section heading. When that happens, the page-level metadata may say `1.3` while the chunk itself starts with recommendation `1.2.7`. The `correct_section_title()` function fixes this.

In [ ]:
# Example: page says section 1.3 but chunk starts with 1.2.7
print(correct_section_title("1.3 Management of local disease", "- 1.2.7 Offer adjuvant"))
print(correct_section_title("1.3 Management of local disease", "- 1.3.1 Offer chemotherapy"))
print(correct_section_title("1.2 Information for people", "- 1.2.1 Discuss treatment"))

## Summary

| Step | What happens | Why it matters |
|------|-------------|----------------|
| Multi-query encoding | 4-5 queries per question | Captures different phrasings |
| Chroma search | `top_k * 4` candidates per query | Enough candidates for reranking |
| Deduplication | Keep highest semantic score per chunk | Same chunk from multiple queries |
| Lexical boost | +0.08 * keyword overlap | Exact word matches get a small boost |
| Intent boost | +0.06 to +0.12 for matching chunks | Symptom questions favor NG12, diagnosis questions favor section 1.2 |
| Section correction | Fix page-boundary heading mismatches | Citations match the actual recommendation |

**Current production settings:** `TOP_K = 5`, `CHUNK_SIZE = 450`, `EMBEDDING_MODEL = multilingual-e5-base`